# Model 1: k-Nearest Neighbors (k-NN)

**Course:** CS280/CS485 — Introduction to Artificial Intelligence, Spring 2026  
**Dataset:** DataCo Smart Supply Chain (Late Delivery Risk Classification)  
**Phase:** 1 of 5 model phases — runs after Phase 0 (Preprocessing)  
**Reference:** `docs.md` Section 5.1 · `PHASES.md` §4.1–4.2

---

This notebook implements k-Nearest Neighbors classification as the first of five architecturally diverse models. k-NN is a non-parametric, instance-based learner: it makes predictions by finding the *k* closest training examples by Euclidean distance and assigning the majority class — requiring no explicit training phase, only storage of the training set.

**Scalability constraint:** Full-dataset k-NN inference has O(n·d) cost per query. On 144K training rows × 232 features, this is computationally prohibitive for a grid search presentation deadline. The fix documented in `docs.md §5.1` is a **stratified 25K subsample** (`replace=False`) used exclusively for k-NN training, while evaluation is done on the full held-out test set.

## 1. Imports

In [ ]:
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils import resample

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Imports complete.')

## 2. Load Preprocessed Data

All Phase 1-5 notebooks load exclusively from `artifacts/prepared_data.pkl` produced by Phase 0 (preprocessing notebook). No raw-CSV access, no refitting of scalers or encoders. This guarantees that every model sees the identical train/test split and feature space.

In [ ]:
with open('../artifacts/prepared_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test  = data['X_test']
y_train = data['y_train']
y_test  = data['y_test']
feature_names = data['feature_names']

print(f'X_train shape : {X_train.shape}')
print(f'X_test  shape : {X_test.shape}')
print(f'y_train class balance — 0: {(y_train==0).sum()}, 1: {(y_train==1).sum()}')
print(f'y_test  class balance — 0: {(y_test==0).sum()},  1: {(y_test==1).sum()}')
print(f'Feature count : {len(feature_names)}')

**Interpretation:** The training set has ~144K rows and 232 features after one-hot encoding and RobustScaler transformation in Phase 0. Class balance is approximately 55% late (1) vs. 45% on-time (0), which is mildly imbalanced — justifying F1-score as the primary tuning metric rather than accuracy.

## 3. Stratified 25K Subsample for k-NN Training

**Why subsampling is necessary:** k-NN has no training phase — the entire training set is stored and queried at inference time. With 144K rows and 232 features, each prediction requires computing 144K Euclidean distances, making a 5-fold GridSearchCV across 30 hyperparameter combinations completely intractable.

**Why this is scientifically valid:**
- The subsample size (~17% of training data) is large enough to preserve the decision boundary landscape.
- Stratification (`stratify=y_train`) preserves the original class ratio exactly — the subsampled minority class is not under-represented.
- `replace=False` ensures no duplicate rows enter training (bootstrapping with replacement would inflate proximity estimates).

This decision is documented in `docs.md §5.1` (Scalability Fix) and must be acknowledged during the presentation.

In [ ]:
# replace=False is CRITICAL — resample() defaults to replace=True (bootstrap).
# For subsampling without duplicates we must set replace=False explicitly.
X_train_knn, y_train_knn = resample(
    X_train, y_train,
    n_samples=25000,
    stratify=y_train,
    replace=False,
    random_state=42
)

print(f'Subsampled training set shape : {X_train_knn.shape}')
print(f'Class balance in subsample — 0: {(y_train_knn==0).sum()}, 1: {(y_train_knn==1).sum()}')
print(f'Class ratio 1/(0+1) in full train  : {y_train.mean():.4f}')
print(f'Class ratio 1/(0+1) in subsample   : {y_train_knn.mean():.4f}')

**Interpretation:** The stratified subsample faithfully preserves the original class ratio (within rounding to whole rows). This confirms that k-NN will not see a distorted class distribution during training — the decision boundary learned from the subsample is a valid proxy for the full-data boundary.

## 4. Hyperparameter Tuning — GridSearchCV

We search over the three hyperparameters specified in `docs.md §5.1`:

| Hyperparameter | Search range | Effect |
|:---|:---|:---|
| `n_neighbors` (k) | {3, 5, 7, 11, 15} | Low k → complex boundary (risk of overfitting); high k → smoother boundary (risk of underfitting) |
| `metric` | {euclidean, manhattan} | Changes the distance definition used to rank neighbors |
| `weights` | {uniform, distance} | Uniform: all k neighbors vote equally; distance: closer neighbors get higher vote weight |

**Total combinations:** 5 × 2 × 2 = 20 fits × 5 CV folds = 100 model evaluations on the 25K subsample.

**Configuration choices:**
- `algorithm='ball_tree'` — builds a spatial index for efficient neighbor lookup (much faster than brute-force on 232-dimensional data).
- `n_jobs=-1` — parallelises across all CPU cores.
- `scoring='f1'` — optimise for F1, which accounts for both precision and recall, appropriate given mild class imbalance.

In [ ]:
param_grid = {
    'n_neighbors': [3, 5, 7, 11, 15],
    'metric':      ['euclidean', 'manhattan'],
    'weights':     ['uniform', 'distance'],
}

knn_base = KNeighborsClassifier(algorithm='ball_tree', n_jobs=-1)

grid_search = GridSearchCV(
    estimator=knn_base,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    return_train_score=True,
    verbose=1
)

print('Starting GridSearchCV (20 combinations × 5 folds = 100 fits on 25K rows)...')
grid_search.fit(X_train_knn, y_train_knn)
print('GridSearchCV complete.')

### 4.1 Best Hyperparameters

In [ ]:
best_params = grid_search.best_params_
best_cv_f1  = grid_search.best_score_

print(f'Best hyperparameters : {best_params}')
print(f'Best CV F1 (validation, 5-fold) : {best_cv_f1:.4f}')

**Interpretation:** The best hyperparameters balance the bias-variance trade-off for this dataset. The chosen k controls boundary smoothness (Lecture 4: low k → high variance; high k → high bias). The metric and weighting scheme that maximise F1 on the 5-fold validation sets within the subsample are selected automatically.

### 4.2 Full GridSearchCV Results Table

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)
cols = ['param_n_neighbors', 'param_metric', 'param_weights',
        'mean_test_score', 'std_test_score', 'rank_test_score']
cv_results_display = (
    cv_results[cols]
    .sort_values('rank_test_score')
    .rename(columns={
        'param_n_neighbors': 'k',
        'param_metric':      'metric',
        'param_weights':     'weights',
        'mean_test_score':   'mean_val_F1',
        'std_test_score':    'std_val_F1',
        'rank_test_score':   'rank',
    })
)
cv_results_display['mean_val_F1'] = cv_results_display['mean_val_F1'].round(4)
cv_results_display['std_val_F1']  = cv_results_display['std_val_F1'].round(4)
print(cv_results_display.to_string(index=False))

**Interpretation:** The table above ranks all 20 hyperparameter combinations by mean 5-fold validation F1. Combinations with high k tend to produce more stable (lower std) but potentially lower F1, while low k can overfit to the local neighborhood. The `distance` weighting generally benefits from the BallTree's efficient distance computation and gives closer neighbors proportionally more influence.

## 5. k vs. F1 Validation Curve

This plot shows how the mean 5-fold validation F1 score varies with k for each (metric, weights) combination. It directly visualises the bias-variance trade-off: small k → complex, potentially overfit boundary; large k → smoother, potentially underfit boundary. This is Figure 8 from `docs.md §7.3.B`.

In [ ]:
k_values = [3, 5, 7, 11, 15]
combos = [
    ('euclidean', 'uniform'),
    ('euclidean', 'distance'),
    ('manhattan', 'uniform'),
    ('manhattan', 'distance'),
]

fig, ax = plt.subplots(figsize=(9, 5))

for metric, weights in combos:
    mask = (
        (cv_results['param_metric'] == metric) &
        (cv_results['param_weights'] == weights)
    )
    subset = cv_results[mask].sort_values('param_n_neighbors')
    f1_means = subset['mean_test_score'].values
    f1_stds  = subset['std_test_score'].values
    label = f'{metric} / {weights}'
    ax.plot(k_values, f1_means, marker='o', label=label)
    ax.fill_between(k_values,
                    f1_means - f1_stds,
                    f1_means + f1_stds,
                    alpha=0.12)

# Mark the best k
best_k = best_params['n_neighbors']
ax.axvline(best_k, color='red', linestyle='--', linewidth=1.2,
           label=f'Best k = {best_k}')

ax.set_xlabel('k (number of neighbors)', fontsize=12)
ax.set_ylabel('Mean 5-fold Validation F1', fontsize=12)
ax.set_title('k-NN: k vs. Validation F1 (GridSearchCV · 25K subsample)', fontsize=13)
ax.xaxis.set_major_locator(mticker.FixedLocator(k_values))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/knn_k_vs_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to results/knn_k_vs_f1.png')

**Interpretation:** The curve shows the trade-off between underfitting (high k, overly smooth boundary) and overfitting (low k, too-sensitive boundary). The optimal k balances these extremes. The shaded band represents ±1 standard deviation across the 5 folds — narrower bands indicate more stable performance. Distance-weighted variants tend to benefit from the smoothing effect of proximity weighting, especially at low k where a single nearest neighbor might otherwise dominate the vote.

## 6. Model Evaluation on Held-Out Test Set

The best estimator returned by `GridSearchCV` was already refit on the full 25K subsampled training set using the best hyperparameters. We now evaluate it on the full held-out test set (36K rows, never seen during training or tuning).

In [ ]:
best_knn = grid_search.best_estimator_

y_pred  = best_knn.predict(X_test)
y_proba = best_knn.predict_proba(X_test)[:, 1]

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_proba)

# Train F1 on the subsampled training set (model was trained on this set)
y_pred_train = best_knn.predict(X_train_knn)
train_f1     = f1_score(y_train_knn, y_pred_train)

print('=== k-NN Test Set Metrics ===')
print(f'Accuracy  : {accuracy:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1-Score  : {f1:.4f}')
print(f'ROC-AUC   : {roc_auc:.4f}')
print(f'Train F1  : {train_f1:.4f}  (on 25K subsample — model\'s own training set)')

**Interpretation:** 
- **Accuracy** provides a baseline but can be misleading when classes are imbalanced (~55% late).
- **F1-Score** is the primary metric: it balances precision (of predicted-late orders, how many were truly late?) and recall (of truly-late orders, how many did we catch?). From the business perspective, missing a late delivery (false negative) is more costly than a false alarm.
- **ROC-AUC** measures ranking quality independent of the classification threshold — a value well above 0.5 indicates the model distinguishes late from on-time orders meaningfully.
- The gap between Train F1 and Test F1 indicates the overfitting level — k-NN with low k may overfit due to its sensitivity to local noise.

## 7. Confusion Matrix

The confusion matrix shows the counts of True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN) on the test set. This is a mandatory deliverable per `docs.md §7.3.B`.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['On-Time (0)', 'Late (1)']
)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('k-NN — Confusion Matrix (Test Set)', fontsize=13)
plt.tight_layout()
plt.savefig('../results/knn_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TN (correct on-time) : {tn:,}')
print(f'FP (false alarm)     : {fp:,}')
print(f'FN (missed late)     : {fn:,}  ← business-critical error')
print(f'TP (correct late)    : {tp:,}')
print('Figure saved to results/knn_confusion_matrix.png')

**Interpretation:** The confusion matrix reveals the distribution of error types:
- **False Negatives (FN)** — the model predicted on-time but the order was late. These are the most costly errors from a business standpoint: logistics teams lose the opportunity to proactively intervene.
- **False Positives (FP)** — the model predicted late but the order arrived on time. These trigger unnecessary interventions (e.g., expedited shipping upgrades) but do not leave the customer with a late delivery.

Given the business objective (early-warning system), a model with low FN is preferred even at the cost of higher FP.

## 8. 5-Fold Stratified Cross-Validation on the Subsampled Training Set

Cross-validation provides an unbiased estimate of generalisation performance and reveals score stability across folds. We run 5-fold Stratified CV on the 25K subsampled training set using the best hyperparameters found by GridSearchCV. Stratification preserves the class ratio in every fold.

In [ ]:
cv_estimator = KNeighborsClassifier(
    algorithm='ball_tree',
    n_jobs=-1,
    **best_params
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_f1_scores = cross_val_score(
    cv_estimator,
    X_train_knn, y_train_knn,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)

print('5-Fold Stratified CV F1 scores (on 25K subsample):')
for i, score in enumerate(cv_f1_scores, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'Mean  : {cv_f1_scores.mean():.4f}')
print(f'Std   : {cv_f1_scores.std():.4f}')

**Interpretation:** The spread of F1 scores across the 5 folds indicates model stability. A low standard deviation (< 0.01) confirms that performance is consistent and not fold-dependent. High variance across folds would suggest the model is sensitive to which 20% of the 25K subsample is held out for validation, which could indicate overfitting to specific neighborhood structures.

## 9. Lecture 4 Connection

### Concept: Nearest-Neighbor Classification

k-NN is the **direct implementation** of the nearest-neighbor classification algorithm introduced in Lecture 4.

---

### Distance Function (Euclidean)

Given a query point **x** and a training point **x**_i with *n* features:

$$d(\mathbf{x}, \mathbf{x_i}) = \sqrt{\sum_{j=1}^{n}(x_j - x_{ij})^2}$$

This is why **RobustScaler** was applied in Phase 0: features on different scales (e.g., price in hundreds vs. discount in [0,1]) would otherwise distort Euclidean distances, making high-magnitude features dominate neighbor selection regardless of their actual predictive relevance.

---

### Majority-Vote Prediction Rule

Given query point **x**, find the set $\mathcal{N}_k(\mathbf{x})$ of the *k* nearest training points. Predict the majority class:

$$\hat{y} = \arg\max_{c} \sum_{\mathbf{x_i} \in \mathcal{N}_k(\mathbf{x})} \mathbf{1}[y_i = c]$$

With `weights='distance'`, the vote of each neighbor is weighted by $1/d(\mathbf{x}, \mathbf{x_i})$, giving closer neighbors proportionally greater influence:

$$\hat{y} = \arg\max_{c} \sum_{\mathbf{x_i} \in \mathcal{N}_k(\mathbf{x})} \frac{\mathbf{1}[y_i = c]}{d(\mathbf{x}, \mathbf{x_i})}$$

---

### Hyperparameter ↔ Lecture Concept Mapping

| sklearn parameter | Lecture 4 concept | Effect |
|:---|:---|:---|
| `n_neighbors` (k) | neighborhood size | Controls bias-variance trade-off: small k → high variance (local overfitting); large k → high bias (over-smoothed boundary) |
| `metric` | distance function d(·,·) | Euclidean = L2 norm; Manhattan = L1 norm — changes which neighbors are "nearest" |
| `weights` | voting rule | Uniform = equal vote; distance = inverse-distance weighting |

---

### Why k-NN Has No Regularization Term

Unlike the Perceptron (`alpha`), SVM (`C`), or XGBoost (`gamma`, `lambda`), k-NN has no explicit regularization parameter mapping to Lecture 4's formula:

$$\text{cost}(h) = \text{loss}(h) + \lambda \cdot \text{complexity}(h)$$

In k-NN, **k itself acts as an implicit regularizer**: increasing k increases the effective neighborhood size, smoothing the decision boundary and reducing model complexity — analogous to increasing λ in the regularization framework.

## 10. Save Results

Save the results dictionary to `results/knn_results.pkl` using the schema defined in `PHASES.md §2.3`. Phase 6 (merge notebook) depends on this exact key set.

In [ ]:
knn_results = {
    'model_name':         'k-NN',
    'model':              best_knn,
    'best_params':        best_params,
    'y_pred':             y_pred,
    'y_proba':            y_proba,
    'metrics': {
        'accuracy':  accuracy,
        'precision': precision,
        'recall':    recall,
        'f1':        f1,
        'roc_auc':   roc_auc,
    },
    'cv_f1_scores':       cv_f1_scores,
    'train_f1':           train_f1,
    'test_f1':            f1,
    'feature_importance': None,  # k-NN is non-parametric — no feature importances
}

# Verify schema keys match PHASES.md §2.3 exactly
required_keys = {
    'model_name', 'model', 'best_params', 'y_pred', 'y_proba',
    'metrics', 'cv_f1_scores', 'train_f1', 'test_f1', 'feature_importance'
}
assert set(knn_results.keys()) == required_keys, 'Schema mismatch!'
assert set(knn_results['metrics'].keys()) == {
    'accuracy', 'precision', 'recall', 'f1', 'roc_auc'
}, 'Metrics sub-schema mismatch!'

with open('../results/knn_results.pkl', 'wb') as f:
    pickle.dump(knn_results, f)

print('knn_results.pkl saved successfully.')
print(f'Keys: {list(knn_results.keys())}')
print(f'Metrics: {knn_results["metrics"]}')

**Verification:** The assert statements confirm the saved dictionary matches the `PHASES.md §2.3` schema exactly. Any deviation would break the Phase 6 merge notebook that loads all five `*_results.pkl` files.

## 11. Summary

| Item | Value |
|:---|:---|
| Algorithm | k-Nearest Neighbors (`ball_tree`, `n_jobs=-1`) |
| Training set | 25K stratified subsample from 144K training rows |
| Hyperparameter search | GridSearchCV (5-fold, F1 scoring, 20 combinations) |
| Best params | See above |
| Test F1 | See above |
| Test ROC-AUC | See above |
| Lecture 4 concept | Nearest-neighbor classification (direct) |
| Feature importance | Not available (instance-based, non-parametric) |

**Expected performance relative to other models (from `docs.md §6`):** k-NN is expected to deliver Low-to-Medium performance. The non-linearity of the supply chain decision boundary means k-NN may capture local patterns, but the high dimensionality (232 features, curse of dimensionality) limits its effectiveness compared to ensemble methods. The performance gap between k-NN and Random Forest/XGBoost quantifies the value of ensemble learning on this specific dataset.